# Phase 1 — SQL Analytics Layer: Business Intelligence Foundation

**Project:** Hospital Operations & Revenue Risk Intelligence Platform  
**Database:** SQLite (production equivalent: MySQL)  
**Goal:** Create a reliable, queryable hospital data layer that leadership can trust for operational and financial decision-making.

---

## Notebook Structure
1. Database Setup & Schema Creation
2. Data Loading & Integrity Constraints
3. Operational Analysis Queries
4. Financial Analysis Queries
5. Data Quality & Integrity Checks

## 1. Database Setup & Schema Creation

> We use SQLite here for portability. The same DDL translates directly to MySQL by replacing `INTEGER PRIMARY KEY AUTOINCREMENT` with `INT AUTO_INCREMENT PRIMARY KEY` and using MySQL's `ENGINE=InnoDB` for FK enforcement.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Styling ──────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

# ── Connect to in-memory SQLite database ─────────────────────────────────────
# NOTE: For MySQL, replace with:
#   import mysql.connector
#   conn = mysql.connector.connect(host='localhost', user='root', password='', database='hospital_db')
DB_PATH = '../hospital.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print('✅ Database connection established.')

In [ ]:
# ── Enable Foreign Key enforcement (SQLite requires explicit activation) ──────
cursor.execute('PRAGMA foreign_keys = ON;')

# ── Drop tables if re-running ─────────────────────────────────────────────────
cursor.executescript('''
    DROP TABLE IF EXISTS billing;
    DROP TABLE IF EXISTS visits;
    DROP TABLE IF EXISTS patients;
''')

# ── Create PATIENTS table ─────────────────────────────────────────────────────
# MySQL equivalent: INT AUTO_INCREMENT PRIMARY KEY, TINYINT(1) for chronic_flag
cursor.execute('''
    CREATE TABLE patients (
        patient_id        INTEGER PRIMARY KEY,
        age               INTEGER NOT NULL CHECK(age >= 0 AND age <= 130),
        gender            VARCHAR(10),
        city              VARCHAR(100),
        insurance_provider VARCHAR(100),
        chronic_flag      INTEGER NOT NULL DEFAULT 0 CHECK(chronic_flag IN (0,1)),
        registration_date DATE
    );
''')

# ── Create VISITS table ───────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE visits (
        visit_id              INTEGER PRIMARY KEY,
        patient_id            INTEGER NOT NULL,
        visit_date            DATE    NOT NULL,
        department            VARCHAR(100),
        visit_type            VARCHAR(50),
        length_of_stay_hours  REAL    CHECK(length_of_stay_hours >= 0),
        risk_score            VARCHAR(20),
        doctor_id             INTEGER,
        FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
    );
''')

# ── Create BILLING table ──────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE billing (
        bill_id         INTEGER PRIMARY KEY,
        visit_id        INTEGER NOT NULL UNIQUE,
        billed_amount   REAL    NOT NULL CHECK(billed_amount >= 0),
        approved_amount REAL    CHECK(approved_amount >= 0),
        claim_status    VARCHAR(20),
        payment_days    INTEGER CHECK(payment_days >= 0),
        billing_date    DATE,
        FOREIGN KEY (visit_id) REFERENCES visits(visit_id)
    );
''')

print('✅ Tables created with PRIMARY KEY, FOREIGN KEY, and CHECK constraints.')

In [ ]:
# ── Create Indexes on commonly queried fields ─────────────────────────────────
# Index justification:
#   visit_date       → time-based filtering in operational reports
#   department       → GROUP BY in department-level aggregations
#   insurance_provider → JOIN & GROUP BY in financial analysis
#   claim_status     → WHERE filtering for revenue leakage queries
#   risk_score       → WHERE filtering for high-risk visit queries

cursor.executescript('''
    CREATE INDEX IF NOT EXISTS idx_visits_date       ON visits(visit_date);
    CREATE INDEX IF NOT EXISTS idx_visits_dept       ON visits(department);
    CREATE INDEX IF NOT EXISTS idx_visits_risk       ON visits(risk_score);
    CREATE INDEX IF NOT EXISTS idx_patients_insurer  ON patients(insurance_provider);
    CREATE INDEX IF NOT EXISTS idx_billing_status    ON billing(claim_status);
    CREATE INDEX IF NOT EXISTS idx_billing_visit     ON billing(visit_id);
''')

print('✅ Indexes created on: visit_date, department, risk_score, insurance_provider, claim_status, visit_id')

## 2. Data Loading

In [ ]:
import os

# ── Load CSVs ─────────────────────────────────────────────────────────────────
BASE = os.path.join(os.path.dirname(os.getcwd()), 'capstone') if 'notebooks' in os.getcwd() else '..'

patients_df = pd.read_csv('../patients.csv', parse_dates=['registration_date'])
visits_df   = pd.read_csv('../visits.csv',   parse_dates=['visit_date'])
billing_df  = pd.read_csv('../billing.csv',  parse_dates=['billing_date'])

# ── Insert into SQLite ────────────────────────────────────────────────────────
patients_df.to_sql('patients', conn, if_exists='append', index=False)
visits_df.to_sql('visits',     conn, if_exists='append', index=False)
billing_df.to_sql('billing',   conn, if_exists='append', index=False)
conn.commit()

print(f'✅ Loaded: {len(patients_df):,} patients | {len(visits_df):,} visits | {len(billing_df):,} billing records')

## 3. Operational Analysis

> **Business context:** Department-level metrics guide staffing, bed allocation, and scheduling decisions for hospital administrators.

In [ ]:
# ── 3.1 Top 10 Departments by Total Visit Volume ──────────────────────────────
q1 = '''
    SELECT department,
           COUNT(*) AS total_visits
    FROM   visits
    GROUP  BY department
    ORDER  BY total_visits DESC
    LIMIT  10;
'''
top_depts = pd.read_sql(q1, conn)

fig, ax = plt.subplots()
sns.barplot(data=top_depts, y='department', x='total_visits', ax=ax, palette='Blues_d')
ax.set_title('Top 10 Departments by Visit Volume')
ax.set_xlabel('Total Visits'); ax.set_ylabel('Department')
for bar in ax.patches:
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('../data_outputs/op_top_depts.png', dpi=110); plt.show()

display(top_depts)

> **📊 Insight:** The top departments account for the majority of patient flow. Departments with the highest volumes require priority resource allocation and real-time capacity monitoring.

In [ ]:
# ── 3.2 Top 5 Departments by Average Length of Stay ───────────────────────────
q2 = '''
    SELECT department,
           ROUND(AVG(length_of_stay_hours), 2) AS avg_los_hours
    FROM   visits
    WHERE  length_of_stay_hours IS NOT NULL
    GROUP  BY department
    ORDER  BY avg_los_hours DESC
    LIMIT  5;
'''
los_depts = pd.read_sql(q2, conn)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=los_depts, y='department', x='avg_los_hours', ax=ax, palette='Oranges_d')
ax.set_title('Top 5 Departments by Avg Length of Stay (Hours)')
ax.set_xlabel('Avg LOS (Hours)'); ax.set_ylabel('Department')
plt.tight_layout(); plt.savefig('../data_outputs/op_avg_los.png', dpi=110); plt.show()
display(los_depts)

> **📊 Insight:** Departments with high average LOS indicate complex case mixes or possible bed-flow bottlenecks. These departments are prime targets for discharge planning and bed management initiatives.

In [ ]:
# ── 3.3 Percentage of High Risk Visits per Department ────────────────────────
q3 = '''
    SELECT department,
           COUNT(*) AS total_visits,
           SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) AS high_risk_visits,
           ROUND(
               100.0 * SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) / COUNT(*), 2
           ) AS high_risk_pct
    FROM   visits
    GROUP  BY department
    ORDER  BY high_risk_pct DESC;
'''
risk_dept = pd.read_sql(q3, conn)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(risk_dept['department'], risk_dept['high_risk_pct'],
               color=sns.color_palette('RdYlGn_r', len(risk_dept)))
ax.set_title('High Risk Visit % by Department')
ax.set_xlabel('High Risk Visit (%)')
ax.axvline(risk_dept['high_risk_pct'].mean(), color='navy', linestyle='--', label='Mean')
ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/op_high_risk_dept.png', dpi=110); plt.show()
display(risk_dept)

> **📊 Insight:** Departments with above-average High Risk percentages require dedicated clinical risk protocols and closer physician oversight. This metric feeds directly into Model A predictions.

In [ ]:
# ── 3.4 Average Number of Visits per Patient by City ─────────────────────────
q4 = '''
    SELECT p.city,
           COUNT(v.visit_id)            AS total_visits,
           COUNT(DISTINCT v.patient_id) AS unique_patients,
           ROUND(
               1.0 * COUNT(v.visit_id) / COUNT(DISTINCT v.patient_id), 2
           ) AS avg_visits_per_patient
    FROM   visits v
    JOIN   patients p ON v.patient_id = p.patient_id
    GROUP  BY p.city
    ORDER  BY avg_visits_per_patient DESC;
'''
city_visits = pd.read_sql(q4, conn)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=city_visits, x='city', y='avg_visits_per_patient', ax=ax, palette='viridis')
ax.set_title('Avg Visits per Patient by City')
ax.set_xlabel('City'); ax.set_ylabel('Avg Visits / Patient')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('../data_outputs/op_city_visits.png', dpi=110); plt.show()
display(city_visits)

In [ ]:
# ── 3.5 Doctors Handling the Most High Risk Visits ────────────────────────────
q5 = '''
    SELECT doctor_id,
           COUNT(*) AS high_risk_visits
    FROM   visits
    WHERE  risk_score = 'High'
    GROUP  BY doctor_id
    ORDER  BY high_risk_visits DESC
    LIMIT  10;
'''
dr_risk = pd.read_sql(q5, conn)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=dr_risk, x='doctor_id', y='high_risk_visits', ax=ax, palette='flare')
ax.set_title('Top 10 Doctors by High Risk Visit Count')
ax.set_xlabel('Doctor ID'); ax.set_ylabel('High Risk Visit Count')
plt.tight_layout(); plt.savefig('../data_outputs/op_dr_risk.png', dpi=110); plt.show()
display(dr_risk)

> **📊 Insight:** Certain doctors consistently handle high volumes of high-risk patients — this may reflect specialisation or case assignment patterns. These physicians should be considered for support resources and workload monitoring.

## 4. Financial Analysis

> **Business context:** These queries power the revenue leakage detection and insurer performance monitoring used by the Finance team.

In [ ]:
# ── 4.1 Top 10 Insurance Providers by Total Billed Amount ────────────────────
q6 = '''
    SELECT p.insurance_provider,
           ROUND(SUM(b.billed_amount), 2)    AS total_billed,
           ROUND(SUM(b.approved_amount), 2)  AS total_approved,
           COUNT(b.bill_id)                  AS total_claims
    FROM   billing b
    JOIN   visits  v ON b.visit_id   = v.visit_id
    JOIN   patients p ON v.patient_id = p.patient_id
    GROUP  BY p.insurance_provider
    ORDER  BY total_billed DESC
    LIMIT  10;
'''
insurer_billed = pd.read_sql(q6, conn)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(insurer_billed))
ax.bar([i - 0.2 for i in x], insurer_billed['total_billed']/1e6, width=0.4, label='Billed', color='steelblue')
ax.bar([i + 0.2 for i in x], insurer_billed['total_approved']/1e6, width=0.4, label='Approved', color='seagreen')
ax.set_xticks(list(x)); ax.set_xticklabels(insurer_billed['insurance_provider'], rotation=30, ha='right')
ax.set_title('Top 10 Insurers: Billed vs Approved Amount (₹ Millions)')
ax.set_ylabel('Amount (₹ Millions)'); ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/fin_insurer_billed.png', dpi=110); plt.show()
display(insurer_billed)

In [ ]:
# ── 4.2 Top 5 Insurance Providers by Claim Rejection Rate ────────────────────
q7 = '''
    SELECT p.insurance_provider,
           COUNT(b.bill_id)  AS total_claims,
           SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END) AS rejected,
           ROUND(
               100.0 * SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END)
               / COUNT(b.bill_id), 2
           ) AS rejection_rate_pct
    FROM   billing  b
    JOIN   visits   v ON b.visit_id   = v.visit_id
    JOIN   patients p ON v.patient_id = p.patient_id
    GROUP  BY p.insurance_provider
    ORDER  BY rejection_rate_pct DESC
    LIMIT  5;
'''
rejection_rate = pd.read_sql(q7, conn)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=rejection_rate, x='insurance_provider', y='rejection_rate_pct', ax=ax, palette='Reds_d')
ax.set_title('Top 5 Insurers by Claim Rejection Rate (%)')
ax.set_xlabel('Insurance Provider'); ax.set_ylabel('Rejection Rate (%)')
ax.axhline(rejection_rate['rejection_rate_pct'].mean(), color='black', linestyle='--', label='Mean')
ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/fin_rejection_rate.png', dpi=110); plt.show()
display(rejection_rate)

> **📊 Insight:** High rejection rates from specific insurers indicate documentation gaps, coding errors, or contractual misalignments. Finance teams should initiate targeted pre-submission audits for these providers.

In [ ]:
# ── 4.3 Average Payment Delay by Insurance Provider ───────────────────────────
q8 = '''
    SELECT p.insurance_provider,
           ROUND(AVG(b.payment_days), 1) AS avg_payment_days,
           COUNT(b.bill_id)              AS paid_claims
    FROM   billing  b
    JOIN   visits   v ON b.visit_id   = v.visit_id
    JOIN   patients p ON v.patient_id = p.patient_id
    WHERE  b.claim_status = 'Paid'
      AND  b.payment_days IS NOT NULL
    GROUP  BY p.insurance_provider
    ORDER  BY avg_payment_days DESC;
'''
payment_delay = pd.read_sql(q8, conn)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=payment_delay, x='insurance_provider', y='avg_payment_days', ax=ax, palette='YlOrRd')
ax.set_title('Avg Payment Delay (Days) by Insurance Provider')
ax.set_xlabel('Insurance Provider'); ax.set_ylabel('Avg Payment Days')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig('../data_outputs/fin_payment_delay.png', dpi=110); plt.show()
display(payment_delay)

In [ ]:
# ── 4.4 Revenue Realization Ratio by Department ───────────────────────────────
# Ratio = approved_amount / billed_amount — measures how much the hospital recovers
q9 = '''
    SELECT v.department,
           ROUND(SUM(b.billed_amount), 2)   AS total_billed,
           ROUND(SUM(b.approved_amount), 2) AS total_approved,
           ROUND(
               100.0 * SUM(b.approved_amount) / NULLIF(SUM(b.billed_amount), 0), 2
           ) AS realization_ratio_pct
    FROM   billing b
    JOIN   visits  v ON b.visit_id = v.visit_id
    GROUP  BY v.department
    ORDER  BY realization_ratio_pct DESC;
'''
rev_ratio = pd.read_sql(q9, conn)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if r >= 70 else '#e74c3c' for r in rev_ratio['realization_ratio_pct']]
ax.barh(rev_ratio['department'], rev_ratio['realization_ratio_pct'], color=colors)
ax.axvline(70, color='black', linestyle='--', label='70% Threshold')
ax.set_title('Revenue Realization Ratio by Department (%)')
ax.set_xlabel('Realization Ratio (%)'); ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/fin_rev_ratio.png', dpi=110); plt.show()
display(rev_ratio)

> **📊 Insight:** Departments below the 70% realization threshold represent revenue leakage hotspots. These should be reviewed for billing accuracy, coding compliance, and contract terms with payers.

In [ ]:
# ── 4.5 High-Billed Visits with Zero or Missing Approved Amount ───────────────
# Threshold: billed_amount above 75th percentile
q10 = '''
    SELECT b.bill_id, b.visit_id, v.department, v.visit_type,
           b.billed_amount, b.approved_amount, b.claim_status
    FROM   billing b
    JOIN   visits  v ON b.visit_id = v.visit_id
    WHERE  (b.approved_amount = 0 OR b.approved_amount IS NULL)
      AND  b.billed_amount > (
               SELECT billed_amount FROM billing
               ORDER BY billed_amount
               LIMIT 1 OFFSET (SELECT CAST(COUNT(*)*0.75 AS INTEGER) FROM billing)
           )
    ORDER  BY b.billed_amount DESC
    LIMIT  20;
'''
high_bill_zero = pd.read_sql(q10, conn)
print(f'🚨 High-value visits with zero/null approved amount: {len(high_bill_zero)}')
display(high_bill_zero.head(10))

## 5. Data Quality & Integrity Checks

> **Business context:** Trust in AI model outputs depends entirely on data integrity. These checks form the foundation of the governance layer.

In [ ]:
# ── 5.1 Visits Without Billing Records (Orphan Visits) ───────────────────────
q11 = '''
    SELECT v.visit_id, v.patient_id, v.visit_date, v.department
    FROM   visits v
    LEFT JOIN billing b ON v.visit_id = b.visit_id
    WHERE  b.visit_id IS NULL;
'''
orphan_visits = pd.read_sql(q11, conn)
print(f'Orphan visits (no billing record): {len(orphan_visits)}')
if len(orphan_visits) > 0:
    display(orphan_visits.head())

In [ ]:
# ── 5.2 Billing Records Without Corresponding Visit ──────────────────────────
q12 = '''
    SELECT b.bill_id, b.visit_id, b.billed_amount, b.claim_status
    FROM   billing b
    LEFT JOIN visits v ON b.visit_id = v.visit_id
    WHERE  v.visit_id IS NULL;
'''
orphan_billing = pd.read_sql(q12, conn)
print(f'Orphan billing records (no visit): {len(orphan_billing)}')

In [ ]:
# ── 5.3 Duplicate Patient IDs ─────────────────────────────────────────────────
q13 = '''
    SELECT patient_id, COUNT(*) AS occurrences
    FROM   patients
    GROUP  BY patient_id
    HAVING COUNT(*) > 1;
'''
dup_patients = pd.read_sql(q13, conn)
print(f'Duplicate patient_id values: {len(dup_patients)}')

In [ ]:
# ── 5.4 Missing or Invalid length_of_stay_hours / payment_days ────────────────
q14 = '''
    SELECT
        SUM(CASE WHEN length_of_stay_hours IS NULL OR length_of_stay_hours < 0 THEN 1 ELSE 0 END)
            AS invalid_los,
        COUNT(*) AS total_visits
    FROM visits;
'''
q15 = '''
    SELECT
        SUM(CASE WHEN payment_days IS NULL THEN 1 ELSE 0 END) AS null_payment_days,
        SUM(CASE WHEN payment_days < 0    THEN 1 ELSE 0 END) AS negative_payment_days,
        COUNT(*) AS total_billing
    FROM billing;
'''
los_check     = pd.read_sql(q14, conn)
payment_check = pd.read_sql(q15, conn)

print('LOS Quality:')
display(los_check)
print('\nPayment Days Quality:')
display(payment_check)

In [ ]:
# ── 5.5 Visits Linked to Patients with Missing Insurance Provider ─────────────
q16 = '''
    SELECT v.visit_id, v.patient_id, v.department, p.insurance_provider
    FROM   visits   v
    JOIN   patients p ON v.patient_id = p.patient_id
    WHERE  p.insurance_provider IS NULL OR TRIM(p.insurance_provider) = '';
'''
missing_insurer = pd.read_sql(q16, conn)
print(f'Visits with missing insurance provider: {len(missing_insurer)}')

# ── Summary Report ────────────────────────────────────────────────────────────
print('\n' + '='*50)
print('DATA QUALITY SUMMARY')
print('='*50)
print(f'  Orphan visits (no billing):         {len(orphan_visits)}')
print(f'  Orphan billing (no visit):          {len(orphan_billing)}')
print(f'  Duplicate patient IDs:              {len(dup_patients)}')
print(f'  Missing/invalid LOS:                {los_check["invalid_los"].values[0]}')
print(f'  Null payment_days:                  {payment_check["null_payment_days"].values[0]}')
print(f'  Visits missing insurance provider:  {len(missing_insurer)}')
print('='*50)

In [ ]:
# ── Save database connection for downstream notebooks ─────────────────────────
conn.close()
print('✅ Phase 1 Complete. Database saved to ../hospital.db')
print('   Next: Phase2_EDA.ipynb for exploratory analysis and feature engineering.')

---

## Phase 1 Summary

| Check | Status |
|---|---|
| Tables created with PKs & FKs | ✅ |
| CHECK constraints on data types | ✅ |
| Indexes on key query columns | ✅ |
| 10 Operational queries | ✅ |
| Financial analysis queries | ✅ |
| Data integrity validation | ✅ |

**Key business flags raised:**
- Null `payment_days` on Pending/Rejected claims — expected but must be handled in modeling
- Some departments show below-70% revenue realization — revenue leakage risk
- High rejection rates concentrated in specific insurers — Finance team action needed